In [ ]:
import re
from pathlib import Path
import pandas as pd

BASE = Path("./backlog")

logs = {
    "BF16 Run A": BASE / "gipfel-train-8b-250s-4n-baseline-35min-1976838.log",
    "BF16 Run B": BASE / "gipfel-train-8b-250s-4n-baseline-35min-1969685.log",
    "BF16 Run C": BASE / "gipfel-train-8b-250s-4n-baseline-35min-1975423.log",
    "FP8 Hybrid": BASE / "gipfel-train-8b-250s-4n-hybrid-35min-1996684.log",
    "FP8 E4M3": BASE / "gipfel-train-8b-250s-4n-E4M3_only-35min-1996685.log",
    "FP8 Optimizer": BASE / "gipfel-train-8b-250s-4n-FP8_stress-35min-1996686.log",
}

pat = re.compile(
    r"iteration\s+(?P<it>\d+)/\s*(?P<total>\d+).*?"
    r"elapsed time per iteration \(ms\):\s+(?P<ms>[\d.]+).*?"
    r"throughput per GPU \(TFLOP/s/GPU\):\s+(?P<tflops>[\d.]+).*?"
    r"tokens/sec/GPU:\s+(?P<tok>[\d.]+).*?"
    r"lm loss:\s+(?P<loss>[\d.Ee+-]+).*?"
    r"grad norm:\s+(?P<grad>[\d.Ee+-]+).*?"
    r"number of skipped iterations:\s+(?P<skip>\d+).*?"
    r"number of nan iterations:\s+(?P<nan>\d+)"
)

mem_pat = re.compile(
    r"\[Rank 0\].*?memory \(MB\).*?"
    r"allocated:\s+(?P<allocated>[\d.]+).*?"
    r"max allocated:\s+(?P<max_allocated>[\d.]+).*?"
    r"reserved:\s+(?P<reserved>[\d.]+).*?"
    r"max reserved:\s+(?P<max_reserved>[\d.]+)"
)

rows = []
mem_rows = []

for name, path in logs.items():
    text = path.read_text(errors="ignore")
    for m in pat.finditer(text):
        rows.append({
            "run": name,
            "iteration": int(m.group("it")),
            "iter_ms": float(m.group("ms")),
            "tflops_gpu": float(m.group("tflops")),
            "tokens_sec_gpu": float(m.group("tok")),
            "lm_loss": float(m.group("loss")),
            "grad_norm": float(m.group("grad")),
            "skipped": int(m.group("skip")),
            "nan": int(m.group("nan")),
        })
    for m in mem_pat.finditer(text):
        mem_rows.append({
            "run": name,
            "allocated_MB": float(m.group("allocated")),
            "max_allocated_MB": float(m.group("max_allocated")),
            "reserved_MB": float(m.group("reserved")),
            "max_reserved_MB": float(m.group("max_reserved")),
        })

df = pd.DataFrame(rows)
mem = pd.DataFrame(mem_rows)

steady = df[(df["iteration"] >= 200) & (df["iteration"] <= 250)]

summary = steady.groupby("run").agg(
    n=("iteration", "count"),
    iter_ms_mean=("iter_ms", "mean"),
    iter_ms_std=("iter_ms", "std"),
    tokens_sec_gpu_mean=("tokens_sec_gpu", "mean"),
    tokens_sec_gpu_std=("tokens_sec_gpu", "std"),
    tflops_gpu_mean=("tflops_gpu", "mean"),
    tflops_gpu_std=("tflops_gpu", "std"),
    final_loss=("lm_loss", "last"),
    final_grad_norm=("grad_norm", "last"),
    skipped_total=("skipped", "sum"),
    nan_total=("nan", "sum"),
)

bf16_mean = summary.loc[["BF16 Run A", "BF16 Run B", "BF16 Run C"], "tokens_sec_gpu_mean"].mean()
summary["speedup_vs_bf16_mean"] = summary["tokens_sec_gpu_mean"] / bf16_mean
summary["time_saved_vs_bf16_mean_%"] = (1 - 1 / summary["speedup_vs_bf16_mean"]) * 100

print("\n=== STEADY-STATE SUMMARY, ITERATIONS 200-250 ===")
print(summary.round(3).to_string())

print("\n=== BF16 BASELINE MEAN ± STD ACROSS RUNS A/B/C ===")
bf = summary.loc[["BF16 Run A", "BF16 Run B", "BF16 Run C"]]
print("tokens/sec/GPU:", round(bf["tokens_sec_gpu_mean"].mean(), 2), "±", round(bf["tokens_sec_gpu_mean"].std(), 2))
print("TFLOP/s/GPU:", round(bf["tflops_gpu_mean"].mean(), 2), "±", round(bf["tflops_gpu_mean"].std(), 2))

if not mem.empty:
    mem_summary = mem.groupby("run").agg(
        max_allocated_MB=("max_allocated_MB", "max"),
        max_reserved_MB=("max_reserved_MB", "max"),
    )
    print("\n=== MEMORY SUMMARY ===")
    print(mem_summary.round(2).to_string())
else:
    print("\nNo memory rows parsed.")



=== STEADY-STATE SUMMARY, ITERATIONS 200-250 ===
                n  iter_ms_mean  iter_ms_std  tokens_sec_gpu_mean  tokens_sec_gpu_std  tflops_gpu_mean  tflops_gpu_std  final_loss  final_grad_norm  skipped_total  nan_total  speedup_vs_bf16_mean  time_saved_vs_bf16_mean_%
run                                                                                                                                                                                                                           
BF16 Run A     51      6285.671       35.431            10426.569              56.658          483.098           2.619       4.880            1.230              0          0                 0.995                     -0.526
BF16 Run B     51      6222.022       34.596            10533.294              56.486          488.041           2.611       4.759            1.016              0          0                 1.005                      0.492
BF16 Run C     51      6250.904       35.138            10

In [ ]:
import re
from pathlib import Path
import pandas as pd

BASE = Path("backlog")  # change to Path("path1-caspar-reduce-floats/backlog") if running from repo root

logs = {
    "BF16 Run A": BASE / "gipfel-train-8b-250s-4n-baseline-35min-1976838.log",
    "BF16 Run B": BASE / "gipfel-train-8b-250s-4n-baseline-35min-1969685.log",
    "BF16 Run C": BASE / "gipfel-train-8b-250s-4n-baseline-35min-1975423.log",
    "FP8 Hybrid": BASE / "gipfel-train-8b-250s-4n-hybrid-35min-1996684.log",
    "FP8 E4M3": BASE / "gipfel-train-8b-250s-4n-E4M3_only-35min-1996685.log",
    "FP8 Optimizer": BASE / "gipfel-train-8b-250s-4n-FP8_stress-35min-1996686.log",
}

# Dense Tensor Core peak throughput per GPU.
# Adjust if your exact GPU/course spec gives different peak values.
PEAK_TFLOPS = {
    "bf16": 989.0,   # H100/GH200 BF16 dense Tensor Core peak
    "fp8": 1979.0,   # H100/GH200 FP8 dense Tensor Core peak
}

WINDOW_START = 200
WINDOW_END = 250

# ---------------------------------------------------------------------
# Parser
# ---------------------------------------------------------------------

pat = re.compile(
    r"iteration\s+(?P<it>\d+)/\s*(?P<total>\d+).*?"
    r"elapsed time per iteration \(ms\):\s+(?P<ms>[\d.]+).*?"
    r"throughput per GPU \(TFLOP/s/GPU\):\s+(?P<tflops>[\d.]+).*?"
    r"tokens/sec/GPU:\s+(?P<tok>[\d.]+).*?"
    r"lm loss:\s+(?P<loss>[\d.Ee+-]+).*?"
    r"grad norm:\s+(?P<grad>[\d.Ee+-]+).*?"
    r"number of skipped iterations:\s+(?P<skip>\d+).*?"
    r"number of nan iterations:\s+(?P<nan>\d+)"
)

rows = []

for run_name, path in logs.items():
    text = path.read_text(errors="ignore")

    precision = "bf16" if run_name.startswith("BF16") else "fp8"
    peak = PEAK_TFLOPS[precision]

    for m in pat.finditer(text):
        tflops = float(m.group("tflops"))

        rows.append({
            "run": run_name,
            "precision_peak_used": precision.upper(),
            "peak_tflops_gpu": peak,
            "iteration": int(m.group("it")),
            "iter_ms": float(m.group("ms")),
            "tokens_sec_gpu": float(m.group("tok")),
            "tflops_gpu": tflops,
            "mfu": tflops / peak,
            "mfu_percent": 100 * tflops / peak,
            "lm_loss": float(m.group("loss")),
            "grad_norm": float(m.group("grad")),
            "skipped": int(m.group("skip")),
            "nan": int(m.group("nan")),
        })

df = pd.DataFrame(rows)

steady = df[
    (df["iteration"] >= WINDOW_START) &
    (df["iteration"] <= WINDOW_END)
].copy()

summary = steady.groupby("run").agg(
    n=("iteration", "count"),
    precision_peak_used=("precision_peak_used", "first"),
    peak_tflops_gpu=("peak_tflops_gpu", "first"),
    tflops_gpu_mean=("tflops_gpu", "mean"),
    tflops_gpu_std=("tflops_gpu", "std"),
    mfu_percent_mean=("mfu_percent", "mean"),
    mfu_percent_std=("mfu_percent", "std"),
    tokens_sec_gpu_mean=("tokens_sec_gpu", "mean"),
    tokens_sec_gpu_std=("tokens_sec_gpu", "std"),
    final_loss=("lm_loss", "last"),
    final_grad_norm=("grad_norm", "last"),
    skipped_total=("skipped", "sum"),
    nan_total=("nan", "sum"),
)

bf16 = summary.loc[["BF16 Run A", "BF16 Run B", "BF16 Run C"]]

bf16_mean_row = pd.DataFrame({
    "n": [bf16["n"].sum()],
    "precision_peak_used": ["BF16"],
    "peak_tflops_gpu": [PEAK_TFLOPS["bf16"]],
    "tflops_gpu_mean": [bf16["tflops_gpu_mean"].mean()],
    "tflops_gpu_std": [bf16["tflops_gpu_mean"].std()],
    "mfu_percent_mean": [bf16["mfu_percent_mean"].mean()],
    "mfu_percent_std": [bf16["mfu_percent_mean"].std()],
    "tokens_sec_gpu_mean": [bf16["tokens_sec_gpu_mean"].mean()],
    "tokens_sec_gpu_std": [bf16["tokens_sec_gpu_mean"].std()],
    "final_loss": [bf16["final_loss"].mean()],
    "final_loss_std": [bf16["final_loss"].std()],
    "final_grad_norm": [bf16["final_grad_norm"].mean()],
    "final_grad_norm_std": [bf16["final_grad_norm"].std()],
    "skipped_total": [bf16["skipped_total"].sum()],
    "nan_total": [bf16["nan_total"].sum()],
}, index=["BF16 Baseline Mean"])

report = pd.concat([bf16_mean_row, summary.loc[["FP8 Hybrid", "FP8 E4M3", "FP8 Optimizer"]]])

print("=== TRUE DTYPE-NATIVE MFU, ITERATIONS 200-250 ===")
display(report.round(3))

print("\nLaTeX-ready rows:")
for run, row in report.iterrows():
    print(
        f"{run} & "
        f"{row['precision_peak_used']} & "
        f"{row['tokens_sec_gpu_mean']:.1f} & "
        f"{row['tflops_gpu_mean']:.1f} & "
        f"{row['peak_tflops_gpu']:.0f} & "
        f"{row['mfu_percent_mean']:.1f}\\% \\\\"
    )

=== TRUE DTYPE-NATIVE MFU, ITERATIONS 200-250 ===


,n,precision_peak_used,peak_tflops_gpu,tflops_gpu_mean,tflops_gpu_std,mfu_percent_mean,mfu_percent_std,tokens_sec_gpu_mean,tokens_sec_gpu_std,final_loss,final_loss_std,final_grad_norm,final_grad_norm_std,skipped_total,nan_total
BF16 Baseline Mean,153,BF16,989.0,485.641,2.475,49.104,0.250,10481.464,53.429,4.853,0.083,1.297,0.319,0,0
FP8 Hybrid,51,FP8,1979.0,720.247,13.912,36.394,0.703,15544.843,300.099,4.847,NaN,0.887,NaN,0,0
FP8 E4M3,51,FP8,1979.0,735.200,14.780,37.150,0.747,15867.529,318.877,7.561,NaN,1.068,NaN,0,0
FP8 Optimizer,51,FP8,1979.0,694.024,15.388,35.069,0.778,14978.882,332.034,144.953,NaN,77.693,NaN,0,0



LaTeX-ready rows:
BF16 Baseline Mean & BF16 & 10481.5 & 485.6 & 989 & 49.1\% \\
FP8 Hybrid & FP8 & 15544.8 & 720.2 & 1979 & 36.4\% \\
FP8 E4M3 & FP8 & 15867.5 & 735.2 & 1979 & 37.2\% \\
FP8 Optimizer & FP8 & 14978.9 & 694.0 & 1979 & 35.1\% \\
